In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","catalog")
dbutils.widgets.text("data_source","customers","Data Source")


In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
base_path =f's3://sports-bar-de/{data_source}/*.csv'
print(base_path)

In [0]:
df=(
    spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path)
    .withColumn('read_timestamp',F.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)

display(df.limit(10))

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")
     


### silver data processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
df_silver = df_bronze.dropDuplicates(['customer_id'])
df_silver.count()

In [0]:
df_silver.head(1)

In [0]:
df_silver = df_silver.withColumn('customer_name',F.trim(F.col("customer_name")))


In [0]:
display(df_silver)

In [0]:
# typos -> correct names
city_mapping = {
    'Bengaluru': 'Bengaluru',
    'Bangalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}
allowed = ['Bengaluru', 'Hyderabad', 'New Delhi']
df_silver = (
    df_silver
    .replace(city_mapping,subset=['city'])
    .withColumn(
        'city',
        F.when(F.col('city').isNull(),None)
        .when(F.col('city').isin(allowed),F.col('city'))
        .otherwise(None)
    )
)
df_silver.select('city').distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    'customer_name',
    F.when(F.col("customer_name").isNull(),None)
    .otherwise(F.initcap(F.col("customer_name")))
)

In [0]:
display(df_silver.select(['customer_id','customer_name','city']).filter(F.col("city").isNull()))

In [0]:
# Business confirmation note: city corrections confirmed by business team
customer_city_fix = {
    #Endurance Foods
    789101: "Bengaluru",
    # Sprintz Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",
    789520:"Bengaluru",
    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)


In [0]:
df_silver = (
    df_silver
        .join(df_fix, "customer_id", "left")
        .withColumn(
            "city",
            F.coalesce("city", "fixed_city")  # Replace null with fixed city
        )
        .drop("fixed_city")
)


In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))

In [0]:
df_silver.printSchema()

In [0]:
df_silver = (df_silver.withColumn(
    "customer",
    F.concat_ws("-","customer_name",F.coalesce(F.col("city"),F.lit("Unknown"))))
    .withColumn("market",F.lit("India")) 
    .withColumn("platform",F.lit("Sports Bar"))
    .withColumn("channel",F.lit("Aquisition"))
) 
    
    

In [0]:
df_silver = df_silver.withColumnRenamed("customer_id", "customer_code")
display(df_silver)


In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
display(df_silver)

### gold processing

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")
display(df_silver)
df_silver = df_silver.drop("customer_id")
display(df_silver)


In [0]:
df_gold = df_silver.select(["customer_code","customer_name","customer","city","market","platform","channel"])

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")


In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")

df_child_customers = (
    spark.table("fmcg.gold.sb_dim_customers").select(
        "customer_code",
        "customer",
        "market",
        "platform",
        "channel"
    )
)


In [0]:
display(df_child_customers)

In [0]:
#upsert operation: it means when customer code match then update or else insert update_insert upsert
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition = "target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
